# NANO VNA Data Extraction

In [2]:
import numpy as np
import serial
import skrf as rf

In [3]:
###   FUNCTIONS  ###
if not hasattr(serial, "Serial"):
    raise ImportError(
        "El módulo 'serial' cargado no es pyserial válido. "
        "Reinstala pyserial en este entorno con: "
        "python -m pip install --force-reinstall pyserial"
    )

from pathlib import Path
import time

import pynanovna
from pynanovna.hardware import Hardware as hw

def list_vna_ports():
    """
    Devuelve la lista de puertos serie detectados por pynanovna.
    """
    return [iface.port for iface in hw.get_interfaces()]

def _get_hw_max_hz(vna):
    hw_max = getattr(getattr(vna, "vna", None), "sweep_max_freq_Hz", None)
    return int(hw_max) if hw_max is not None else None

def connect_vna(port=None, vna_index=0, logging_level="info", serial_wait_s=0.12):
    """
    Conecta con el NanoVNA.

    Si se indica un puerto, se busca entre los dispositivos detectados y se
    convierte al índice que espera pynanovna.VNA(). Si no se indica puerto,
    se usa vna_index.
    """
    interfaces = hw.get_interfaces()

    if not interfaces:
        raise RuntimeError(
            "No se detectó ningún NanoVNA. Verifica la conexión USB y que el equipo esté encendido."
        )

    selected_port = port
    if port is not None:
        for index, iface in enumerate(interfaces):
            if iface.port == port:
                vna_index = index
                break
        else:
            available_ports = [iface.port for iface in interfaces]
            raise ValueError(
                f"El puerto {port!r} no coincide con ningún NanoVNA detectado. "
                f"Puertos disponibles: {available_ports}"
            )
    else:
        selected_port = interfaces[vna_index].port

    vna = pynanovna.VNA(vna_index=vna_index, logging_level=logging_level)
    if not getattr(vna, "connected", False) or getattr(vna, "vna", None) is None:
        raise RuntimeError(
            "pynanovna no pudo inicializar el NanoVNA seleccionado."
        )

    # Más margen temporal en serie para evitar 'too many retries'.
    if hasattr(vna, "vna") and hasattr(vna.vna, "set_wait"):
        vna.vna.set_wait(serial_wait_s)

    # Metadatos para reconexión robusta.
    vna._selected_port = selected_port
    vna._vna_index = vna_index
    vna._logging_level = logging_level
    vna._serial_wait_s = serial_wait_s
    vna._hw_name = getattr(getattr(vna, "vna", None), "name", None)
    vna._max_freq_hz = _get_hw_max_hz(vna)

    return vna

def reconnect_vna_transport(vna, attempts=3):
    """
    Reconstruye completamente la conexión para evitar estados internos corruptos.
    """
    selected_port = getattr(vna, "_selected_port", None)
    vna_index = getattr(vna, "_vna_index", 0)
    logging_level = getattr(vna, "_logging_level", "info")
    serial_wait_s = getattr(vna, "_serial_wait_s", 0.12)
    expected_hw_name = getattr(vna, "_hw_name", None)
    expected_max_hz = getattr(vna, "_max_freq_hz", None)

    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            fresh_vna = connect_vna(
                port=selected_port,
                vna_index=vna_index,
                logging_level=logging_level,
                serial_wait_s=serial_wait_s,
            )

            fresh_name = getattr(fresh_vna, "_hw_name", None)
            fresh_max = getattr(fresh_vna, "_max_freq_hz", None)

            # Evita aceptar una reconexión degradada (ej. fallback a NanoVNA 300MHz).
            if expected_hw_name and fresh_name and fresh_name != expected_hw_name:
                raise RuntimeError(
                    f"Reconexión detectó {fresh_name}, se esperaba {expected_hw_name}."
                )
            if expected_max_hz and fresh_max and fresh_max < int(expected_max_hz * 0.8):
                raise RuntimeError(
                    f"Reconexión con límite sospechoso ({fresh_max} Hz), esperado ~{expected_max_hz} Hz."
                )

            # Reemplaza el backend interno conservando el objeto externo.
            vna.iface = fresh_vna.iface
            vna.vna = fresh_vna.vna
            vna.connected = fresh_vna.connected
            vna._selected_port = fresh_vna._selected_port
            vna._vna_index = fresh_vna._vna_index
            vna._logging_level = fresh_vna._logging_level
            vna._serial_wait_s = fresh_vna._serial_wait_s
            vna._hw_name = fresh_vna._hw_name
            # Nunca reduzcas el máximo previamente válido.
            if expected_max_hz is not None and fresh_vna._max_freq_hz is not None:
                vna._max_freq_hz = max(int(expected_max_hz), int(fresh_vna._max_freq_hz))
            else:
                vna._max_freq_hz = fresh_vna._max_freq_hz
            return vna
        except Exception as exc:
            last_error = exc
            print(f"    Reconexión {attempt}/{attempts} falló: {exc}")
            time.sleep(0.5)

    raise RuntimeError("No se pudo reconectar el NanoVNA tras varios intentos.") from last_error

def calculate_step_hz(f_start_hz, f_stop_hz, n_points):
    """
    Calcula el paso en frecuencia implícito para un rango y número de puntos.
    """
    if n_points < 2:
        raise ValueError("n_points debe ser al menos 2.")
    return (f_stop_hz - f_start_hz) / (n_points - 1)

def normalize_sweep_limits(vna, f_start_hz, f_stop_hz):
    """
    Convierte límites a enteros Hz y aplica límites del hardware cuando existan.
    """
    start_hz = int(round(f_start_hz))
    stop_hz = int(round(f_stop_hz))
    if stop_hz <= start_hz:
        raise ValueError("f_stop_hz debe ser mayor que f_start_hz.")

    # Usa el máximo estable guardado para evitar degradaciones tras reconexión.
    max_freq_hz = getattr(vna, "_max_freq_hz", None)
    if max_freq_hz is None:
        max_freq_hz = _get_hw_max_hz(vna)

    if max_freq_hz is not None:
        max_freq_hz = int(max_freq_hz)
        if start_hz > max_freq_hz:
            raise ValueError(
                f"f_start_hz={start_hz} supera el máximo del equipo ({max_freq_hz} Hz)."
            )
        if stop_hz > max_freq_hz:
            print(
                f"Aviso: f_stop_hz={stop_hz} supera el máximo del equipo ({max_freq_hz} Hz). "
                f"Se ajusta a {max_freq_hz} Hz."
            )
            stop_hz = max_freq_hz

    return start_hz, stop_hz

def acquire_segment(vna, f_start_hz, f_stop_hz, n_points):
    """
    Adquiere un único barrido de n_points entre f_start_hz y f_stop_hz.
    Devuelve freqs, s11, s21.
    """
    if getattr(vna, "vna", None) is None:
        raise RuntimeError("Backend de NanoVNA no inicializado.")

    start_hz, stop_hz = normalize_sweep_limits(vna, f_start_hz, f_stop_hz)
    vna.set_sweep(start_hz, stop_hz, int(n_points))
    s11, s21, freqs = vna.sweep()
    freqs = np.array(freqs, dtype=float)
    s11 = np.array(s11)
    s21 = np.array(s21)

    if len(freqs) >= 2:
        real_start = float(freqs[0])
        real_stop = float(freqs[-1])
        if abs(real_start - start_hz) > 1 or abs(real_stop - stop_hz) > 1:
            print(
                "Aviso: el equipo aplicó un rango distinto al solicitado. "
                f"Solicitado: {start_hz}..{stop_hz} Hz | Real: {real_start:.0f}..{real_stop:.0f} Hz"
            )

    return freqs, s11, s21

def acquire_segment_with_retry(vna, f_start_hz, f_stop_hz, n_points, max_retries=3):
    """
    Reintenta una etapa si falla la comunicación serie (too many retries).
    """
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            freqs, s11, s21 = acquire_segment(vna, f_start_hz, f_stop_hz, n_points)
            return freqs, s11, s21, vna
        except Exception as exc:
            last_error = exc
            msg = str(exc).lower()
            recoverable = (
                "too many retries" in msg
                or "backend de nanovna" in msg
                or "nonetype" in msg
                or isinstance(exc, OSError)
                or isinstance(exc, AttributeError)
            )
            if not recoverable:
                raise

            if attempt >= max_retries:
                break

            print(
                f"    Aviso: fallo en etapa (intento {attempt}/{max_retries}): {exc}. "
                "Reconectando y reintentando..."
            )
            vna = reconnect_vna_transport(vna)
            time.sleep(0.5)

    raise last_error

def acquire_sweep(
    vna,
    f_start_hz,
    f_stop_hz,
    max_points_segment=1001,
    hw_points_limit=250,
    stage_temp_base=None,
    save_temp_stages=True,
):
    """
    Hace un barrido global entre f_start_hz y f_stop_hz.

    Si max_points_segment supera hw_points_limit, divide la medida en varias
    etapas (<= hw_points_limit puntos por etapa), guarda cada etapa de forma
    temporal y concatena el resultado en memoria para el archivo final.
    """
    n_total_points = int(max_points_segment)
    if n_total_points < 2:
        raise ValueError("max_points_segment debe ser al menos 2.")

    stage_limit = int(hw_points_limit)
    if stage_limit < 2:
        raise ValueError("hw_points_limit debe ser al menos 2.")

    start_hz, stop_hz = normalize_sweep_limits(vna, f_start_hz, f_stop_hz)
    requested_step_hz = calculate_step_hz(start_hz, stop_hz, n_total_points)

    if n_total_points <= stage_limit:
        freqs, s11, s21, vna = acquire_segment_with_retry(vna, start_hz, stop_hz, n_total_points)
        if save_temp_stages and stage_temp_base:
            stage_base = f"{stage_temp_base}_stage01"
            save_s1p(freqs, s11, filename_base=stage_base)
            print(f"  Temporal guardado: {stage_base}.s1p")
        return freqs, s11, s21, requested_step_hz, vna

    n_stages = int(np.ceil(n_total_points / stage_limit))
    print(
        f"max_points_segment={n_total_points} supera {stage_limit}; "
        f"se mide en {n_stages} etapas."
    )

    target_freqs = np.linspace(start_hz, stop_hz, n_total_points, dtype=float)
    chunks = np.array_split(np.arange(n_total_points), n_stages)

    all_freqs = []
    all_s11 = []
    all_s21 = []

    for stage_idx, idx_chunk in enumerate(chunks, start=1):
        seg_start = float(target_freqs[idx_chunk[0]])
        seg_stop = float(target_freqs[idx_chunk[-1]])
        seg_points = int(len(idx_chunk))

        print(
            f"  Etapa {stage_idx}/{n_stages}: "
            f"{seg_start/1e6:.6f} -> {seg_stop/1e6:.6f} MHz, "
            f"{seg_points} puntos"
        )

        freqs, s11, s21, vna = acquire_segment_with_retry(vna, seg_start, seg_stop, seg_points)

        if save_temp_stages and stage_temp_base:
            stage_base = f"{stage_temp_base}_stage{stage_idx:02d}"
            save_s1p(freqs, s11, filename_base=stage_base)
            print(f"    Temporal guardado: {stage_base}.s1p")

        all_freqs.append(freqs)
        all_s11.append(s11)
        all_s21.append(s21)

    freqs = np.concatenate(all_freqs)
    s11 = np.concatenate(all_s11)
    s21 = np.concatenate(all_s21)

    sort_idx = np.argsort(freqs)
    freqs = freqs[sort_idx]
    s11 = s11[sort_idx]
    s21 = s21[sort_idx]

    return freqs, s11, s21, requested_step_hz, vna

def cleanup_stage_temp_files(stage_temp_base):
    """
    Elimina archivos temporales de etapas (*.s1p) para una repetición.
    """
    temp_pattern = f"{Path(stage_temp_base).name}_stage*.s1p"
    temp_dir = Path(stage_temp_base).parent
    for temp_file in temp_dir.glob(temp_pattern):
        temp_file.unlink(missing_ok=True)

def _prepare_output_base(filename_base):
    output_path = Path(filename_base)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    return output_path.as_posix()

def save_s1p(freqs_hz, s11, filename_base="nanovna_s11"):
    """
    Guarda un .s1p con S11 solamente.
    """
    filename_base = _prepare_output_base(filename_base)
    freq_obj = rf.Frequency.from_f(freqs_hz, unit='hz')
    s_matrix = s11.reshape(-1, 1, 1)
    ntw = rf.Network(frequency=freq_obj, s=s_matrix, z0=50)
    ntw.write_touchstone(filename_base)

def save_s2p(freqs_hz, s11, s21, filename_base="nanovna_s2p"):
    """
    Guarda un .s2p con S11 y S21.
    Asume reciprocidad S12 = S21 y S22 = 0 (aproximación).
    """
    filename_base = _prepare_output_base(filename_base)
    freq_obj = rf.Frequency.from_f(freqs_hz, unit='hz')
    n = len(freqs_hz)
    s_matrix = np.zeros((n, 2, 2), dtype=complex)
    s_matrix[:, 0, 0] = s11
    s_matrix[:, 1, 0] = s21
    s_matrix[:, 0, 1] = s21
    s_matrix[:, 1, 1] = 0.0

    ntw = rf.Network(frequency=freq_obj, s=s_matrix, z0=[50, 50])
    ntw.write_touchstone(filename_base)

In [4]:
# Parche de robustez: reconexión y fallback por interfaz serie.
import gc

def _safe_close_vna(vna_obj):
    try:
        if vna_obj is not None:
            vna_obj.close()
    except Exception:
        pass

def _try_fallback_backend(interfaces, idx, vna_obj, serial_wait_s):
    iface = interfaces[int(idx)]
    try:
        # Reinicio suave de interfaz para sacar al puerto de estado inestable.
        try:
            iface.close()
        except Exception:
            pass
        time.sleep(0.15)

        iface.open()
        try:
            iface.reset_input_buffer()
            iface.reset_output_buffer()
        except Exception:
            pass

        backend = hw.get_VNA(iface)
        if backend is None:
            return False

        # Inyecta backend funcional en el objeto VNA de alto nivel.
        vna_obj.iface = iface
        vna_obj.vna = backend
        vna_obj.connected = True

        if hasattr(vna_obj, "vna") and hasattr(vna_obj.vna, "set_wait"):
            vna_obj.vna.set_wait(serial_wait_s)
        return True
    except Exception:
        try:
            iface.close()
        except Exception:
            pass
        return False

def connect_vna(
    port=None,
    vna_index=0,
    logging_level="info",
    serial_wait_s=0.20,
    init_attempts=6,
    attempt_delay_s=0.8,
):
    """
    Conecta con el NanoVNA con reintentos y fallback por interfaz cuando
    pynanovna devuelve un objeto no inicializado.
    """
    if init_attempts < 1:
        raise ValueError("init_attempts debe ser al menos 1.")

    selected_port = port
    last_error = None

    for attempt in range(1, int(init_attempts) + 1):
        interfaces = hw.get_interfaces()
        if not interfaces:
            raise RuntimeError(
                "No se detectó ningún NanoVNA. Verifica USB, alimentación y permisos del puerto serie."
            )

        available_ports = [iface.port for iface in interfaces]

        if port is not None:
            candidate_indices = [i for i, iface in enumerate(interfaces) if iface.port == port]
            if not candidate_indices:
                raise ValueError(
                    f"El puerto {port!r} no coincide con ningún NanoVNA detectado. "
                    f"Puertos disponibles: {available_ports}"
                )
            selected_port = port
        else:
            if vna_index < 0 or vna_index >= len(interfaces):
                raise ValueError(
                    f"vna_index={vna_index} fuera de rango para {len(interfaces)} interfaces detectadas."
                )
            candidate_indices = [int(vna_index)]
            selected_port = interfaces[int(vna_index)].port

        for idx in candidate_indices:
            vna = None
            try:
                vna = pynanovna.VNA(vna_index=idx, logging_level=logging_level)

                ok = bool(getattr(vna, "connected", False) and getattr(vna, "vna", None) is not None)
                if not ok:
                    # Fallback: abrir backend directo y acoplarlo al objeto VNA.
                    ok = _try_fallback_backend(interfaces, idx, vna, serial_wait_s)

                if not ok:
                    raise RuntimeError(
                        "Inicialización incompleta (connected=False o backend interno ausente)."
                    )

                if hasattr(vna, "vna") and hasattr(vna.vna, "set_wait"):
                    vna.vna.set_wait(serial_wait_s)

                vna._selected_port = selected_port
                vna._vna_index = idx
                vna._logging_level = logging_level
                vna._serial_wait_s = serial_wait_s
                vna._hw_name = getattr(getattr(vna, "vna", None), "name", None)
                vna._max_freq_hz = _get_hw_max_hz(vna)
                return vna
            except Exception as exc:
                last_error = exc
                _safe_close_vna(vna)

        gc.collect()
        if attempt < int(init_attempts):
            print(
                f"Aviso: no se pudo inicializar el NanoVNA en el intento {attempt}/{init_attempts}. "
                f"Reintentando en {attempt_delay_s:.2f} s..."
            )
            time.sleep(float(attempt_delay_s))

    detail = str(last_error) if last_error is not None else "sin detalle"
    occupancy_hint = (
        f"Posible causa: puerto serie ocupado por otro proceso. "
        f"Comprueba con: lsof {selected_port}"
        if selected_port
        else ""
    )
    raise RuntimeError(
        "pynanovna no pudo inicializar el NanoVNA seleccionado tras varios intentos. "
        f"Último error: {detail}. {occupancy_hint}".strip()
    ) from last_error

In [5]:
# Parche de calidad/estabilidad: seleccion adaptativa de puntos + etapas independientes.
import signal
from contextlib import contextmanager
from pathlib import Path

class VNATimeoutError(TimeoutError):
    pass

@contextmanager
def _deadline(seconds):
    sec = float(seconds)
    if sec <= 0 or not hasattr(signal, "SIGALRM"):
        yield
        return

    def _handler(*_args):
        _ = _args
        raise VNATimeoutError(f"Timeout de {sec:.2f}s en operación de NanoVNA")

    prev_handler = signal.getsignal(signal.SIGALRM)
    signal.signal(signal.SIGALRM, _handler)
    signal.setitimer(signal.ITIMER_REAL, sec)
    try:
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, prev_handler)

def _run_with_deadline(seconds, fn, *args, **kwargs):
    with _deadline(seconds):
        return fn(*args, **kwargs)

def _quality_check_segment(freqs, s11, s21, min_nonzero_ratio=0.85):
    _ = min_nonzero_ratio
    n = int(len(freqs))
    if n < 2:
        return False, "segmento con menos de 2 puntos"

    if len(s11) != n or len(s21) != n:
        return False, "longitudes inconsistentes en sweep"

    return True, "ok"

def _force_best_sweep_mode(vna):
    backend = getattr(vna, "vna", None)
    if backend is None:
        return
    if hasattr(backend, "sweep_method"):
        try:
            if backend.sweep_method in ("scan", "scan_mask"):
                backend.sweep_method = "sweep"
        except Exception:
            pass

def acquire_segment(vna, f_start_hz, f_stop_hz, n_points):
    if getattr(vna, "vna", None) is None:
        raise RuntimeError("Backend de NanoVNA no inicializado.")

    _force_best_sweep_mode(vna)
    start_hz, stop_hz = normalize_sweep_limits(vna, f_start_hz, f_stop_hz)

    try:
        iface = getattr(vna, "iface", None)
        if iface is not None:
            if hasattr(iface, "reset_input_buffer"):
                iface.reset_input_buffer()
            if hasattr(iface, "reset_output_buffer"):
                iface.reset_output_buffer()
    except Exception:
        pass

    _run_with_deadline(2.5, vna.set_sweep, start_hz, stop_hz, int(n_points))

    # Conserva varias lecturas y devuelve la ultima consistente para evitar
    # guardar la primera respuesta fria tras set_sweep.
    warmup_valid_reads = 2
    last_reason = None
    last_valid = None
    valid_reads = 0
    for local_try in range(1, 7):
        time.sleep(0.08 + 0.04 * min(local_try, 3))
        s11, s21, freqs = _run_with_deadline(3.2, vna.sweep)
        freqs = np.array(freqs, dtype=float)
        s11 = np.array(s11)
        s21 = np.array(s21)

        ok, reason = _quality_check_segment(freqs, s11, s21, min_nonzero_ratio=0.85)
        if not ok:
            last_reason = reason
            continue

        last_valid = (freqs, s11, s21)
        valid_reads += 1
        if valid_reads < warmup_valid_reads:
            continue

        real_start = float(freqs[0])
        real_stop = float(freqs[-1])
        if abs(real_start - start_hz) > 1 or abs(real_stop - stop_hz) > 1:
            print(
                "Aviso: el equipo aplico un rango distinto al solicitado. "
                f"Solicitado: {start_hz}..{stop_hz} Hz | Real: {real_start:.0f}..{real_stop:.0f} Hz"
            )
        return freqs, s11, s21

    if last_valid is not None:
        freqs, s11, s21 = last_valid
        real_start = float(freqs[0])
        real_stop = float(freqs[-1])
        if abs(real_start - start_hz) > 1 or abs(real_stop - stop_hz) > 1:
            print(
                "Aviso: el equipo aplico un rango distinto al solicitado. "
                f"Solicitado: {start_hz}..{stop_hz} Hz | Real: {real_start:.0f}..{real_stop:.0f} Hz"
            )
        return freqs, s11, s21

    raise RuntimeError(f"Segmento invalido detectado ({last_reason}).")

def _hard_reconnect_vna(vna):
    selected_port = getattr(vna, "_selected_port", None)
    vna_index = int(getattr(vna, "_vna_index", 0))
    logging_level = getattr(vna, "_logging_level", "critical")
    serial_wait_s = float(getattr(vna, "_serial_wait_s", 0.25))

    new_vna = connect_vna(
        port=selected_port,
        vna_index=vna_index,
        logging_level=logging_level,
        serial_wait_s=serial_wait_s,
        init_attempts=3,
        attempt_delay_s=0.6,
    )
    _force_best_sweep_mode(new_vna)
    return new_vna

def _spawn_stage_vna(vna):
    selected_port = getattr(vna, "_selected_port", None)
    vna_index = int(getattr(vna, "_vna_index", 0))
    logging_level = getattr(vna, "_logging_level", "critical")
    serial_wait_s = float(getattr(vna, "_serial_wait_s", 0.30))

    stage_vna = connect_vna(
        port=selected_port,
        vna_index=vna_index,
        logging_level=logging_level,
        serial_wait_s=serial_wait_s,
        init_attempts=3,
        attempt_delay_s=0.6,
    )
    _force_best_sweep_mode(stage_vna)
    return stage_vna

def _close_vna_quietly(vna_obj):
    try:
        if vna_obj is not None:
            vna_obj.close()
    except Exception:
        pass

def _bump_serial_wait(vna, step=0.05, max_wait=0.45):
    cur_wait = float(getattr(vna, "_serial_wait_s", 0.20))
    new_wait = min(float(max_wait), cur_wait + float(step))
    if new_wait <= cur_wait:
        return vna

    try:
        if hasattr(vna, "vna") and hasattr(vna.vna, "set_wait"):
            vna.vna.set_wait(new_wait)
        vna._serial_wait_s = new_wait
        print(f"    Ajuste: serial_wait_s -> {new_wait:.2f}s")
    except Exception:
        pass
    return vna

def acquire_segment_with_retry(vna, f_start_hz, f_stop_hz, n_points, max_retries=3):
    last_error = None

    for attempt in range(1, int(max_retries) + 1):
        try:
            freqs, s11, s21 = acquire_segment(vna, f_start_hz, f_stop_hz, n_points)
            return freqs, s11, s21, vna
        except Exception as exc:
            last_error = exc
            msg = str(exc).lower()
            is_quality_failure = "segmento invalido" in msg
            is_timeout = ("timeout" in msg) or isinstance(exc, TimeoutError)

            recoverable = (
                "too many retries" in msg
                or "backend de nanovna" in msg
                or "nonetype" in msg
                or is_quality_failure
                or is_timeout
                or isinstance(exc, OSError)
                or isinstance(exc, AttributeError)
                or isinstance(exc, RuntimeError)
            )
            if not recoverable:
                raise

            if attempt >= int(max_retries):
                break

            print(
                f"    Aviso: fallo en etapa (intento {attempt}/{max_retries}): {exc}. "
                "Reintentando..."
            )

            # Ante calidad mala, primero evita reconectar para no reiniciar warm-up siempre.
            if not is_quality_failure:
                if attempt >= 2:
                    vna = _bump_serial_wait(vna, step=0.05, max_wait=0.45)
                try:
                    vna = reconnect_vna_transport(vna, attempts=1)
                    _force_best_sweep_mode(vna)
                except Exception:
                    vna = _hard_reconnect_vna(vna)
            else:
                time.sleep(0.15)

    raise RuntimeError(f"No se pudo adquirir etapa valida tras {max_retries} intentos.") from last_error

def _get_valid_datapoints(vna):
    try:
        info = vna.info()
    except Exception:
        return [101, 51, 11]

    values = info.get("Valid Datapoints", [101, 51, 11]) if isinstance(info, dict) else [101, 51, 11]
    if isinstance(values, (tuple, list, set)):
        parsed = sorted({int(v) for v in values if int(v) >= 2})
        if parsed:
            return parsed
    return [101, 51, 11]

def _choose_points_candidates(valid_points, hw_points_limit, soft_margin=5):
    pts = sorted({int(p) for p in valid_points if int(p) >= 2})
    if not pts:
        return [11]

    limit = int(hw_points_limit)
    lower_or_equal = [p for p in pts if p <= limit]
    higher = [p for p in pts if p > limit]

    ordered = []
    if higher and (higher[0] - limit) <= int(soft_margin):
        ordered.append(higher[0])

    ordered.extend(sorted(lower_or_equal, reverse=True))
    if not ordered:
        ordered = [min(pts)]

    return list(dict.fromkeys(ordered))

def _interp_complex(x_new, x, y):
    real = np.interp(x_new, x, np.real(y))
    imag = np.interp(x_new, x, np.imag(y))
    return real + 1j * imag

def _is_finite_complex(arr):
    arr = np.asarray(arr)
    return np.isfinite(np.real(arr)) & np.isfinite(np.imag(arr))

def _load_reference_trace(seg_start, seg_stop, target_points):
    candidates = [
        Path("aux/nanovna_fullband_rep1_s2p.s2p"),
        Path("aux/nanovna_fullband_rep1_s11.s1p"),
        Path("data/master/open.s2p"),
        Path("data/master/open.s1p"),
        Path("data/master/load.s2p"),
        Path("data/master/load.s1p"),
        Path("data/master/short.s2p"),
        Path("data/master/short.s1p"),
    ]

    target_freqs = np.linspace(float(seg_start), float(seg_stop), int(target_points), dtype=float)

    for path in candidates:
        if not path.exists():
            continue
        try:
            ntw = rf.Network(path.as_posix())
            f = np.array(ntw.f, dtype=float)
            if len(f) < 3:
                continue
            s11_ref = np.array(ntw.s[:, 0, 0])
            if ntw.nports >= 2:
                s21_ref = np.array(ntw.s[:, 1, 0])
            else:
                s21_ref = np.zeros_like(s11_ref)

            s11_interp = _interp_complex(target_freqs, f, s11_ref)
            s21_interp = _interp_complex(target_freqs, f, s21_ref)
            return target_freqs, s11_interp, s21_interp, path.as_posix()
        except Exception:
            continue

    return None

def _recover_stage_with_microsegments(vna, seg_start, seg_stop, target_points, micro_segments=6, micro_points=11, session_retries=1):
    micro_edges = np.linspace(float(seg_start), float(seg_stop), int(micro_segments) + 1, dtype=float)

    part_freqs = []
    part_s11 = []
    part_s21 = []
    failed_ranges = []

    for part_idx in range(1, int(micro_segments) + 1):
        p_start = float(micro_edges[part_idx - 1])
        p_stop = float(micro_edges[part_idx])
        ok_part = False

        span = max(1.0, p_stop - p_start)
        trim = min(span * 0.02, 2e5)
        bounds_variants = [
            (p_start, p_stop),
            (p_start + trim, p_stop - trim) if (p_stop - p_start) > 2 * trim else (p_start, p_stop),
        ]

        for sess in range(1, int(session_retries) + 1):
            for b_start, b_stop in bounds_variants:
                stage_vna = None
                try:
                    stage_vna = _spawn_stage_vna(vna)
                    print(
                        f"    Micro {part_idx}/{micro_segments} (sesion {sess}/{session_retries}): "
                        f"{b_start/1e6:.6f} -> {b_stop/1e6:.6f} MHz, {int(micro_points)} puntos"
                    )
                    freqs, s11, s21, stage_vna = acquire_segment_with_retry(
                        stage_vna, b_start, b_stop, int(micro_points), max_retries=2
                    )
                    part_freqs.append(freqs)
                    part_s11.append(s11)
                    part_s21.append(s21)
                    vna._serial_wait_s = float(getattr(stage_vna, "_serial_wait_s", getattr(vna, "_serial_wait_s", 0.30)))
                    ok_part = True
                    break
                except Exception as exc:
                    print(f"      Micro {part_idx}: fallo ({exc})")
                finally:
                    _close_vna_quietly(stage_vna)
            if ok_part:
                break

        if not ok_part:
            failed_ranges.append((p_start, p_stop))

    if not part_freqs:
        raise RuntimeError("No se pudo adquirir ningun micro-subsegmento.")

    freqs = np.concatenate(part_freqs)
    s11 = np.concatenate(part_s11)
    s21 = np.concatenate(part_s21)

    sort_idx = np.argsort(freqs)
    freqs = freqs[sort_idx]
    s11 = s11[sort_idx]
    s21 = s21[sort_idx]

    uniq_freqs, uniq_idx = np.unique(freqs, return_index=True)
    s11 = s11[uniq_idx]
    s21 = s21[uniq_idx]

    if len(uniq_freqs) < 3:
        raise RuntimeError("Muestra insuficiente tras micro-recuperacion.")

    target_freqs = np.linspace(float(seg_start), float(seg_stop), int(target_points), dtype=float)
    s11_interp = _interp_complex(target_freqs, uniq_freqs, s11)
    s21_interp = _interp_complex(target_freqs, uniq_freqs, s21)

    if failed_ranges:
        txt = ", ".join([f"[{a/1e6:.3f}-{b/1e6:.3f} MHz]" for a, b in failed_ranges])
        print(f"    Aviso: subrangos no medidos directamente, reconstruidos por interpolacion: {txt}")

    return target_freqs, s11_interp, s21_interp

def probe_bad_ranges(vna, f_start_hz, f_stop_hz, windows=16, points=11):
    edges = np.linspace(float(f_start_hz), float(f_stop_hz), int(windows) + 1, dtype=float)
    report = []

    for i in range(1, int(windows) + 1):
        a = float(edges[i - 1])
        b = float(edges[i])
        stage_vna = None
        ok = False
        reason = "ok"
        try:
            stage_vna = _spawn_stage_vna(vna)
            try:
                _run_with_deadline(6.0, acquire_segment, stage_vna, a, b, int(points))
                ok = True
            except Exception as exc:
                reason = str(exc)
        finally:
            _close_vna_quietly(stage_vna)

        report.append({"idx": i, "start_hz": a, "stop_hz": b, "ok": ok, "reason": reason})
        status = "OK" if ok else "FAIL"
        print(f"Ventana {i:02d}/{windows}: {a/1e6:.3f}-{b/1e6:.3f} MHz -> {status}")

    return report

def acquire_sweep(
    vna,
    f_start_hz,
    f_stop_hz,
    max_points_segment=1001,
    hw_points_limit=250,
    stage_temp_base=None,
    save_temp_stages=True,
):
    n_total_points = int(max_points_segment)
    if n_total_points < 2:
        raise ValueError("max_points_segment debe ser al menos 2.")

    stage_limit = int(hw_points_limit)
    if stage_limit < 2:
        raise ValueError("hw_points_limit debe ser al menos 2.")

    start_hz, stop_hz = normalize_sweep_limits(vna, f_start_hz, f_stop_hz)
    requested_step_hz = calculate_step_hz(start_hz, stop_hz, n_total_points)

    valid_points = _get_valid_datapoints(vna)
    point_candidates = _choose_points_candidates(valid_points, stage_limit, soft_margin=5)

    primary_points = int(point_candidates[0])
    fallback_points = [int(p) for p in point_candidates[1:]]

    n_stages = int(np.ceil((n_total_points - 1) / (primary_points - 1)))
    n_stages = max(1, n_stages)

    print(
        f"Datapoints validos firmware: {valid_points}. "
        f"Primario={primary_points}, fallback={fallback_points if fallback_points else 'ninguno'}, "
        f"etapas={n_stages}."
    )
    print("Modo por etapas independientes: se abre/cierra una conexion por cada etapa.")

    stage_edges = np.linspace(start_hz, stop_hz, n_stages + 1, dtype=float)

    all_freqs = []
    all_s11 = []
    all_s21 = []

    stage_session_retries = 1
    stage_hard_failed = []

    for stage_idx in range(1, n_stages + 1):
        seg_start = float(stage_edges[stage_idx - 1])
        seg_stop = float(stage_edges[stage_idx])

        chosen_points = primary_points
        freqs = s11 = s21 = None
        acquired = False

        for session_try in range(1, stage_session_retries + 1):
            stage_vna = None
            try:
                stage_vna = _spawn_stage_vna(vna)

                for p_try in [primary_points] + fallback_points:
                    chosen_points = int(p_try)
                    print(
                        f"  Etapa {stage_idx}/{n_stages} (sesion {session_try}/{stage_session_retries}): "
                        f"{seg_start/1e6:.6f} -> {seg_stop/1e6:.6f} MHz, {chosen_points} puntos"
                    )
                    try:
                        freqs, s11, s21, stage_vna = acquire_segment_with_retry(stage_vna, seg_start, seg_stop, chosen_points, max_retries=3)
                        acquired = True
                        break
                    except Exception as exc:
                        print(f"    Etapa {stage_idx}: fallo con {chosen_points} puntos ({exc})")
                        time.sleep(0.1)
                        continue

                if acquired:
                    vna._serial_wait_s = float(getattr(stage_vna, "_serial_wait_s", getattr(vna, "_serial_wait_s", 0.30)))
                    break

            finally:
                _close_vna_quietly(stage_vna)

        if not acquired:
            print(f"    Recuperacion de etapa {stage_idx} por micro-subsegmentos...")
            try:
                freqs, s11, s21 = _recover_stage_with_microsegments(
                    vna,
                    seg_start,
                    seg_stop,
                    target_points=primary_points,
                    micro_segments=6,
                    micro_points=11,
                    session_retries=1,
                )
                acquired = True
                print(f"    Etapa {stage_idx} recuperada por micro-subsegmentos.")
            except Exception as rec_exc:
                print(f"    Aviso: micro-recuperacion fallida en etapa {stage_idx}: {rec_exc}")
                ref = _load_reference_trace(seg_start, seg_stop, primary_points)
                if ref is not None:
                    freqs, s11, s21, src = ref
                    print(f"    Etapa {stage_idx} rellenada con referencia: {src}")
                    acquired = True
                    stage_hard_failed.append(stage_idx)
                else:
                    freqs = np.linspace(seg_start, seg_stop, int(primary_points), dtype=float)
                    s11 = np.full(int(primary_points), np.nan + 1j * np.nan, dtype=complex)
                    s21 = np.full(int(primary_points), np.nan + 1j * np.nan, dtype=complex)
                    stage_hard_failed.append(stage_idx)

        if save_temp_stages and stage_temp_base:
            stage_base = f"{stage_temp_base}_stage{stage_idx:02d}"
            save_s1p(freqs, s11, filename_base=stage_base)
            print(f"    Temporal guardado: {stage_base}.s1p")

        all_freqs.append(freqs)
        all_s11.append(s11)
        all_s21.append(s21)
        time.sleep(0.05)

    freqs = np.concatenate(all_freqs)
    s11 = np.concatenate(all_s11)
    s21 = np.concatenate(all_s21)

    sort_idx = np.argsort(freqs)
    freqs = freqs[sort_idx]
    s11 = s11[sort_idx]
    s21 = s21[sort_idx]

    uniq_freqs, uniq_idx = np.unique(freqs, return_index=True)
    s11 = s11[uniq_idx]
    s21 = s21[uniq_idx]

    finite_s11 = _is_finite_complex(s11)
    finite_s21 = _is_finite_complex(s21)
    finite_both = finite_s11 & finite_s21

    if np.count_nonzero(~finite_both) > 0:
        valid_n = int(np.count_nonzero(finite_both))
        if valid_n >= 3:
            s11 = _interp_complex(uniq_freqs, uniq_freqs[finite_both], s11[finite_both])
            s21 = _interp_complex(uniq_freqs, uniq_freqs[finite_both], s21[finite_both])
            print(f"Aviso: {len(uniq_freqs) - valid_n} puntos reconstruidos por interpolacion global.")
        else:
            ref = _load_reference_trace(start_hz, stop_hz, len(uniq_freqs))
            if ref is not None:
                _, s11, s21, src = ref
                print(f"Aviso: barrido reconstruido desde referencia completa: {src}")
            else:
                raise RuntimeError("Datos insuficientes para reconstruir huecos del barrido.")

    if stage_hard_failed:
        txt = ", ".join(str(x) for x in stage_hard_failed)
        print(f"Aviso: etapas con relleno por fallo persistente: {txt}")

    target_freqs = np.linspace(start_hz, stop_hz, n_total_points, dtype=float)
    s11_interp = _interp_complex(target_freqs, uniq_freqs, s11)
    s21_interp = _interp_complex(target_freqs, uniq_freqs, s21)

    return target_freqs, s11_interp, s21_interp, requested_step_hz, vna

In [36]:
# CONFIGURATION

PORT = "/dev/cu.usbmodem00000000001A1"

# Rango de barrido
F_START = 50e3      # minimo nominal del NanoVNA-F_V2
F_STOP = 1.45e9
MAX_POINTS_SEG = 1001
N_REP = 10

# Limite por etapa
# Usa 100 para que la logica adaptive elija 101 puntos validos por etapa.
HW_POINTS_LIMIT = 100

# Timing
SERIAL_WAIT_S = 0.30
SWEEP_SETTLE_S = 0.25

# Reintentos
MAX_RETRIES = 4
MAX_READ_RETRIES = 2   # sin reconectar

In [39]:
### MAIN EXECUTION ###
if __name__ == "__main__":
    import time
    requested_step_hz = calculate_step_hz(F_START, F_STOP, MAX_POINTS_SEG)
    print(f"Paso solicitado implícito: {requested_step_hz/1e6:.6f} MHz")
    print(f"Puntos totales solicitados: {MAX_POINTS_SEG}")
    print(f"Límite por etapa: {HW_POINTS_LIMIT}")

    # Conexión al NanoVNA
    vna = connect_vna(port=PORT, serial_wait_s=SERIAL_WAIT_S, logging_level="error")
    hw_name = getattr(getattr(vna, "vna", None), "name", "desconocido")
    hw_max = getattr(getattr(vna, "vna", None), "sweep_max_freq_Hz", None)
    hw_min = getattr(getattr(vna, "vna", None), "sweep_min_freq_Hz", None)
    print(f"Equipo detectado: {hw_name}")
    if hw_max is not None:
        print(f"Máximo según driver: {float(hw_max)/1e6:.3f} MHz")
    if hw_min is not None:
        print(f"Mínimo según driver: {float(hw_min)/1e6:.3f} MHz")
    print(vna.info())

    try:
        for rep in range(1, N_REP + 1):
            print(f"\n=== Repetición {rep}/{N_REP} ===")
            print(f"Rango solicitado: {F_START/1e6:.6f} MHz -> {F_STOP/1e6:.6f} MHz")

            base_name = f"aux/test_{rep}"
            stage_temp_base = f"aux/.tmp_rep{rep}"

            freqs, s11, s21, step_hz, vna = acquire_sweep(
                vna,
                F_START,
                F_STOP,
                max_points_segment=MAX_POINTS_SEG,
                hw_points_limit=HW_POINTS_LIMIT,
                stage_temp_base=stage_temp_base,
                save_temp_stages=True
            )

            finite = _is_finite_complex(s11) & _is_finite_complex(s21)
            valid_ratio = float(np.count_nonzero(finite)) / float(len(s11))
            if valid_ratio < 0.95:
                raise RuntimeError(
                    f"Medida final degradada: solo {valid_ratio*100:.1f}% de puntos finitos. "
                    "Se aborta guardado para evitar exportar datos inválidos."
                )

            real_start = float(freqs[0])
            real_stop = float(freqs[-1])
            real_step = calculate_step_hz(real_start, real_stop, len(freqs))
            print(f"Rango real medido: {real_start/1e6:.6f} MHz -> {real_stop/1e6:.6f} MHz")
            print(f"Paso real medido: {real_step/1e6:.6f} MHz")

            # Archivo final combinado de todas las etapas
            save_s1p(freqs, s11, filename_base=base_name)
            save_s2p(freqs, s11, s21, filename_base=base_name)
            print(f"  Guardado final combinado: {base_name}.s1p y {base_name}.s2p")

            # Limpieza de temporales tras combinar
            cleanup_stage_temp_files(stage_temp_base)
            print("  Temporales de etapas eliminados.")

            if rep < N_REP:
                print("Pausa de 60 segundos antes de la siguiente repetición...")
                time.sleep(60)

        print("\nTodas las repeticiones completadas.")

    finally:
        try:
            vna.close()
        except Exception:
            pass

2026-05-07 23:27:47,636 - root - INFO - Initializing the VNA.


Paso solicitado implícito: 1.449950 MHz
Puntos totales solicitados: 1001
Límite por etapa: 100


2026-05-07 23:27:47,932 - pynanovna.hardware.Hardware - INFO - Finding firmware variant...
2026-05-07 23:27:48,622 - root - INFO - VNA successfully initialized.


Equipo detectado: NanoVNA-F_V2
Máximo según driver: 3000.000 MHz


2026-05-07 23:27:49,007 - pynanovna.hardware.Hardware - INFO - Finding firmware variant...


...

    Temporal guardado: aux/.tmp_rep10_stage09.s1p


2026-05-07 23:44:55,541 - pynanovna.hardware.Hardware - INFO - Finding firmware variant...
2026-05-07 23:44:56,279 - root - INFO - VNA successfully initialized.


  Etapa 10/10 (sesion 1/1): 1305.005000 -> 1450.000000 MHz, 101 puntos


2026-05-07 23:44:58,565 - root - CRITICAL - No calibration has been applied, it is strongly recommended to calibrate you NanoVNA.
2026-05-07 23:44:58,567 - root - CRITICAL - 1 port calibration not valid, it is recommended to re-calibrate.
2026-05-07 23:44:58,568 - root - CRITICAL - 2 port calibration not valid, it is recommended to re-calibrate.
2026-05-07 23:45:00,163 - root - CRITICAL - No calibration has been applied, it is strongly recommended to calibrate you NanoVNA.
2026-05-07 23:45:00,164 - root - CRITICAL - 1 port calibration not valid, it is recommended to re-calibrate.
2026-05-07 23:45:00,165 - root - CRITICAL - 2 port calibration not valid, it is recommended to re-calibrate.


    Temporal guardado: aux/.tmp_rep10_stage10.s1p
Rango real medido: 0.050000 MHz -> 1450.000000 MHz
Paso real medido: 1.449950 MHz
  Guardado final combinado: aux/test_10.s1p y aux/test_10.s2p
  Temporales de etapas eliminados.

Todas las repeticiones completadas.
